# AeroTwin-DT — Digital-Twin Model Training (Colab, GPU)

**AeroTwin-DT** is a synchronised digital twin of the *AE-115T*, a
4-cylinder horizontally-opposed, turbo-normalised ~85 kW aero piston engine of
the class used on MALE UAV platforms. The twin runs the same physics plant
model in three places (backend service, edge node, browser), and this notebook
trains the five learned modules that sit on top of it.

## Problem statement

On-wing piston engines degrade through several competing mechanisms (ring/bore
wear, injector clogging, radiator fouling, oil degradation, bearing wear,
sensor drift, ignition wear). Raw telemetry alone is dominated by the flight
condition, not by health. We therefore
1. run a *healthy shadow twin* alongside the engine and use the
   **residuals** (measured − twin) as physics-informed features, and
2. learn five heads on a run-to-failure corpus generated by the plant model.

## Architecture

```
         mission profile ──► plant model (degrading)  ──┐
                          └► plant model (healthy)    ──┤ residuals
                                                        ▼
   28 features = 16 raw channels + 11 residuals + t (hours since overhaul)
        │
        ├─ M1 anomaly     Conv1D/LSTM sequence auto-encoder   → novelty score
        ├─ M2 diagnosis   CNN + GRU classifier                → 8 fault classes
        ├─ M3 prognosis   physics-informed RUL net (PINN)     → hours to failure
        ├─ M4 forecast    multi-head trend forecaster         → 7 components, +100 h
        └─ M5 advisory    priority head                       → ROUTINE…IMMEDIATE
```

## What each module does

| # | Module | Deep model (this notebook) | Distilled student (deployed) |
|---|--------|----------------------------|------------------------------|
| 1 | Anomaly detection | Conv1D encoder + LSTM decoder sequence AE, threshold = 99.5th pct of healthy reconstruction error | dense 28-16-8-16-28, tanh |
| 2 | Fault classification | Conv1D feature extractor + GRU, class-balanced | dense 27-64-32-8, relu |
| 3 | RUL regression | physics-informed net: MSE + monotonicity penalty + degradation-consistency loss | dense 28-48-24-1, relu |
| 4 | Degradation forecast | shared trunk + 7 heads, 100 engine-hour horizon | dense 28-32-7, relu |
| 5 | Maintenance advisory | priority classifier over (value, rate, forecast, RUL, anomaly, confidence) | dense 6-24-4, relu |

Everything is finally **distilled into the small dense students above** and
exported as `dt_models.json` (schema 2) so the identical weights execute in the
NumPy backend runtime, on the edge node (ONNX/INT8) and in the browser.

> **Evaluation protocol:** the corpus is split by **engine unit** (20 % of units
> held out, whole units only). Rows inside one engine are heavily
> autocorrelated — a random row split inflates every metric.

## 0. Environment

In [ ]:
# Colab already ships TensorFlow; the rest is for ONNX export / quantisation.
# (subprocess rather than the `!pip` magic so every cell is plain, parseable Python)
import subprocess, sys
PKGS = ['tensorflow', 'tf2onnx', 'onnx', 'onnxruntime', 'scikit-learn',
        'seaborn', 'pyarrow']
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS],
                     capture_output=True, text=True).stdout[-2000:])

In [ ]:
import os, json, math, time, itertools, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(tf.test.gpu_device_name())
else:
    print('WARNING: no GPU. Runtime > Change runtime type > Hardware accelerator = GPU')
SEED = 11
np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Physics plant model (inlined)

The cell below is a **verbatim copy of `backend/app/sim/engine.py`**, embedded by
`ml/build_notebook.py` at generation time so the notebook is self-contained on
Colab. Do not hand-edit it — change the backend file and regenerate the
notebook, otherwise the twin and the training corpus drift apart.

In [ ]:
# --------------------------------------------------------------------------
# INLINED VERBATIM FROM backend/app/sim/engine.py BY ml/build_notebook.py
# Do not edit here: change the backend file and regenerate the notebook.
# --------------------------------------------------------------------------
"""
AeroTwin-DT :: Physics-informed aero piston engine simulator
============================================================
Reference engine : 4-cylinder horizontally-opposed, liquid/air cooled,
                   turbo-normalised, ~85 kW class (Rotax 914 / P&W class
                   analogue) used on MALE UAV platforms.

The same equations are mirrored 1:1 in  web/js/engine-model.js  so that the
browser Digital Twin, the edge service and the training pipeline all share a
single source of physical truth (this is what makes the twin *synchronised*).

Units: SI unless noted.  RPM in rev/min, temps in degC, pressures in kPa/bar,
fuel flow in kg/h, vibration in g-RMS.
"""
from __future__ import annotations
import math
import numpy as np

# ----------------------------------------------------------------------------
# Engine constants (indigenous MALE-UAV piston engine, "AE-115T")
# ----------------------------------------------------------------------------
ENGINE = dict(
    displacement_l=1.211,      # total swept volume [L]
    cylinders=4,
    idle_rpm=1400.0,
    max_rpm=5800.0,
    max_power_kw=84.5,
    compression_ratio=9.0,
    lhv_mj_kg=43.5,            # lower heating value of AVGAS/MOGAS
    stoich_afr=14.7,
    boost_limit_bar=1.38,      # wastegate limited MAP
    crit_alt_m=4900.0,         # turbo critical altitude
    oil_capacity_l=3.0,
    prop_gear_ratio=2.43,
)
R_AIR = 287.05


# ----------------------------------------------------------------------------
# Atmosphere (ISA + deviation)
# ----------------------------------------------------------------------------
def isa(alt_m: float, isa_dev_c: float = 0.0):
    """Return (static pressure [Pa], static temperature [K], density [kg/m3])."""
    t = 288.15 - 0.0065 * min(alt_m, 11000.0) + isa_dev_c
    p = 101325.0 * (1.0 - 2.25577e-5 * min(alt_m, 11000.0)) ** 5.25588
    rho = p / (R_AIR * t)
    return p, t, rho


# ----------------------------------------------------------------------------
# Health / degradation state vector
# ----------------------------------------------------------------------------
HEALTH_KEYS = [
    "ring_wear",        # blow-by, compression loss
    "injector_clog",    # fuel delivery restriction on cyl #k
    "cooling_fouling",  # radiator/fin fouling -> CHT rise
    "oil_degradation",  # viscosity loss -> oil pressure drop
    "bearing_wear",     # vibration signature growth
    "sensor_drift",     # EGT/CHT probe drift
    "ignition_wear",    # plug/coil erosion -> misfire probability
]


def fresh_health():
    return {k: 0.0 for k in HEALTH_KEYS}


class AeroPistonEngine:
    """Continuous-time engine plant model with first-order thermal dynamics."""

    def __init__(self, health=None, seed: int | None = None):
        self.h = dict(fresh_health()) if health is None else dict(health)
        self.rng = np.random.default_rng(seed)
        self.rpm = ENGINE["idle_rpm"]
        self.cht = 40.0
        self.egt = 200.0
        self.oil_t = 35.0
        self.coolant_t = 40.0
        self.battery_soc = 1.0
        self.hours = 0.0
        self.misfire_latch = 0.0
        # per-unit relative damage rate for each degradation mode (fleet spread)
        self.rate_mult = {k: 1.0 for k in HEALTH_KEYS}

    # -- steady state maps ---------------------------------------------------
    def _map_kpa(self, throttle, p_amb, alt_m):
        """Manifold absolute pressure with turbo-normalisation + wastegate."""
        na = p_amb * (0.32 + 0.68 * throttle)                       # naturally aspirated part
        boost_avail = ENGINE["boost_limit_bar"] * 1e5 * min(1.0, max(0.0, 1.15 - alt_m / (ENGINE["crit_alt_m"] * 1.9)))
        boosted = min(boost_avail, na * (1.0 + 1.05 * throttle))
        return max(28.0, boosted / 1000.0)                          # kPa

    def _vol_eff(self, rpm):
        x = rpm / ENGINE["max_rpm"]
        return 0.78 + 0.30 * x - 0.26 * x * x                       # peaked VE curve

    def step(self, dt, throttle, alt_m, isa_dev_c, tas_ms=45.0, deg_rate=1.0):
        """Advance the plant by dt seconds. Returns a telemetry dict."""
        h = self.h
        p_amb, t_amb_k, rho = isa(alt_m, isa_dev_c)
        t_amb_c = t_amb_k - 273.15

        # --- rotational dynamics (first order lag toward governed RPM) -------
        rpm_cmd = ENGINE["idle_rpm"] + (ENGINE["max_rpm"] - ENGINE["idle_rpm"]) * throttle
        rpm_cmd *= (1.0 - 0.06 * h["ring_wear"])
        tau_rpm = 0.55
        self.rpm += (rpm_cmd - self.rpm) * (1.0 - math.exp(-dt / tau_rpm))
        self.rpm = float(np.clip(self.rpm + self.rng.normal(0, 4.0), 900, 6200))

        # --- charge / breathing ---------------------------------------------
        map_kpa = self._map_kpa(throttle, p_amb, alt_m)
        ve = self._vol_eff(self.rpm) * (1.0 - 0.22 * h["ring_wear"])
        t_intake = t_amb_k + 26.0 * (map_kpa * 1000.0 / p_amb - 1.0)   # post-compressor rise
        m_air = (map_kpa * 1000.0 * ENGINE["displacement_l"] * 1e-3 * ve
                 * (self.rpm / 60.0 / 2.0)) / (R_AIR * t_intake)        # kg/s

        # --- fuelling -------------------------------------------------------
        lam_target = 1.18 - 0.30 * throttle                             # rich at high power
        afr = ENGINE["stoich_afr"] * lam_target
        m_fuel = m_air / afr * (1.0 - 0.35 * h["injector_clog"])
        fuel_flow_kgh = m_fuel * 3600.0

        # --- misfire / combustion stability ---------------------------------
        # fraction of firing events that misfire (rate-based, dt independent)
        p_misfire = 0.0015 + 0.42 * h["ignition_wear"] ** 1.6 + 0.30 * h["injector_clog"] ** 1.8
        p_misfire *= 1.0 + 0.8 * max(0.0, (alt_m - 5500.0) / 6000.0)
        misfire = float(np.clip(p_misfire + self.rng.normal(0, 0.01), 0.0, 1.0))
        tau_m = max(6.0, dt)
        self.misfire_latch = max(misfire, self.misfire_latch * math.exp(-dt / tau_m))
        comb_eff = 0.985 - 0.20 * self.misfire_latch - 0.05 * h["ring_wear"]

        # --- power ----------------------------------------------------------
        eta_th = (1.0 - ENGINE["compression_ratio"] ** -0.36) * 0.56   # ~0.30 BTE, bsfc ~280 g/kWh
        power_kw = m_fuel * ENGINE["lhv_mj_kg"] * 1e3 * eta_th * comb_eff
        power_kw = float(np.clip(power_kw, 0.0, ENGINE["max_power_kw"] * 1.08))
        bsfc = (fuel_flow_kgh / max(power_kw, 1e-3)) * 1000.0           # g/kWh

        # --- exhaust gas temperature ----------------------------------------
        egt_eq = (t_amb_c + 300.0 + 620.0 * (power_kw / ENGINE["max_power_kw"]) ** 0.62
                  - 140.0 * (lam_target - 1.0) + 90.0 * self.misfire_latch)
        self.egt += (egt_eq - self.egt) * (1.0 - math.exp(-dt / 3.5))

        # --- cylinder head temperature (thermal RC network) ------------------
        cooling = (0.50 + 0.50 * min(1.0, tas_ms / 50.0)) * (rho / 1.225) ** 0.35
        cooling *= (1.0 - 0.42 * h["cooling_fouling"])
        cht_eq = t_amb_c + (45.0 + 80.0 * (power_kw / ENGINE["max_power_kw"])) / max(cooling, 0.45)
        cht_eq = max(cht_eq, 92.0)   # thermostat / cowl-flap governed floor
        self.cht += (cht_eq - self.cht) * (1.0 - math.exp(-dt / 26.0))

        # --- oil circuit ------------------------------------------------------
        oil_eq = 0.62 * self.cht + 18.0 + 8.0 * h["oil_degradation"]
        self.oil_t += (oil_eq - self.oil_t) * (1.0 - math.exp(-dt / 55.0))
        visc = math.exp(-0.021 * (self.oil_t - 90.0)) * (1.0 - 0.30 * h["oil_degradation"])
        oil_p = (1.05 + 3.6 * (self.rpm / ENGINE["max_rpm"]) ** 0.85) * min(1.35, max(0.45, visc))
        oil_p *= (1.0 - 0.22 * h["bearing_wear"])
        oil_p = float(max(0.25, oil_p + self.rng.normal(0, 0.02)))       # bar

        # --- coolant ----------------------------------------------------------
        self.coolant_t += ((0.80 * self.cht + 12.0) - self.coolant_t) * (1.0 - math.exp(-dt / 40.0))

        # --- vibration signature ---------------------------------------------
        base_vib = 0.28 + 0.85 * (self.rpm / ENGINE["max_rpm"]) ** 1.6
        vib = (base_vib * (1.0 + 1.9 * h["bearing_wear"])
               + 1.6 * self.misfire_latch + 0.9 * h["ring_wear"] * (self.rpm / 4000.0))
        vib = float(max(0.05, vib + self.rng.normal(0, 0.03)))           # g RMS
        vib_1x = vib * (0.55 + 0.9 * h["bearing_wear"])                  # 1x order band
        vib_half = vib * (0.10 + 1.3 * self.misfire_latch)               # 0.5x -> misfire band

        # --- electrical --------------------------------------------------------
        alt_load = 0.45 + 0.35 * throttle
        alt_v = 13.9 + 0.35 * min(1.0, self.rpm / 2600.0) - 0.9 * h["ignition_wear"]
        alt_i = 18.0 * alt_load * (1.0 - 0.25 * h["ignition_wear"])
        charging = alt_v > 13.2 and self.rpm > 1800
        self.battery_soc = float(np.clip(self.battery_soc + (0.00004 if charging else -0.00012) * dt, 0.05, 1.0))
        bus_v = alt_v if charging else 12.2 + 1.4 * self.battery_soc

        # --- injection timing / ECU ---------------------------------------------
        inj_cmd = 12.0 + 16.0 * (self.rpm / ENGINE["max_rpm"]) - 4.0 * (map_kpa / 140.0)
        inj_act = inj_cmd - 3.5 * h["injector_clog"] + self.rng.normal(0, 0.15)
        inj_pw = 2.2 + 6.0 * throttle * (1.0 + 0.4 * h["injector_clog"])   # ms

        # --- sensor drift (fault injected on the *measurement*) ------------------
        drift = h["sensor_drift"]
        cht_m = self.cht * (1.0 + 0.06 * drift) + self.rng.normal(0, 0.6)
        egt_m = self.egt * (1.0 - 0.05 * drift) + self.rng.normal(0, 4.0)

        # --- degradation progression --------------------------------------------
        stress = (0.35 + 0.65 * (power_kw / ENGINE["max_power_kw"])) * (1.0 + max(0.0, (self.cht - 165.0) / 70.0))
        inc = deg_rate * stress * dt / 3600.0            # equivalent damaging hours
        h["ring_wear"] = min(1.0, h["ring_wear"] + inc * 6.0e-4 * self.rate_mult["ring_wear"])
        h["bearing_wear"] = min(1.0, h["bearing_wear"] + inc * 3.6e-4 * self.rate_mult["bearing_wear"] * (1 + 2 * vib_1x))
        h["cooling_fouling"] = min(1.0, h["cooling_fouling"] + inc * 4.8e-4 * self.rate_mult["cooling_fouling"])
        h["oil_degradation"] = min(1.0, h["oil_degradation"] + inc * 8.5e-4 * self.rate_mult["oil_degradation"])
        h["injector_clog"] = min(1.0, h["injector_clog"] + inc * 5.8e-4 * self.rate_mult["injector_clog"])
        h["ignition_wear"] = min(1.0, h["ignition_wear"] + inc * 7.0e-4 * self.rate_mult["ignition_wear"])
        h["sensor_drift"] = min(1.0, h["sensor_drift"] + inc * 3.0e-4 * self.rate_mult["sensor_drift"])
        self.hours += dt / 3600.0

        # per-cylinder spread (mirrors web/js/engine-model.js — drives the 3D twin
        # and the cylinder-balance view; cylinder 3 carries the misfire)
        _bias = (1.00, 1.03, 0.97, 1.01)
        cyl_egt, cyl_cht = [], []
        for i in range(4):
            sick = self.misfire_latch if i == 2 else 0.15 * self.misfire_latch
            cyl_egt.append(egt_m * _bias[i] - 120.0 * sick + self.rng.normal(0, 3.0))
            cyl_cht.append(cht_m * _bias[i] + 6.0 * sick + self.rng.normal(0, 0.8))

        return dict(
            cylEGT=cyl_egt, cylCHT=cyl_cht,
            t=self.hours, rpm=self.rpm, throttle=throttle, alt_m=alt_m, oat_c=t_amb_c,
            map_kpa=map_kpa, fuel_flow=fuel_flow_kgh, power_kw=power_kw, bsfc=bsfc,
            egt=egt_m, cht=cht_m, oil_p=oil_p, oil_t=self.oil_t, coolant_t=self.coolant_t,
            vib=vib, vib_1x=vib_1x, vib_half=vib_half, bus_v=bus_v, alt_i=alt_i,
            soc=self.battery_soc, inj_cmd=inj_cmd, inj_act=inj_act, inj_pw=inj_pw,
            misfire=self.misfire_latch, lam=lam_target, **{f"h_{k}": h[k] for k in HEALTH_KEYS},
        )


# ----------------------------------------------------------------------------
# Mission profile generator
# ----------------------------------------------------------------------------
MISSION_PHASES = [
    # (name, duration_s, throttle, alt_m, tas)
    ("START",   90,  0.15,   200, 0),
    ("TAXI",   180,  0.22,   200, 8),
    ("TAKEOFF", 120, 0.98,   400, 32),
    ("CLIMB",  1500, 0.88,  5200, 42),
    ("CRUISE", 3600, 0.62,  7600, 58),
    ("LOITER", 7200, 0.48,  7000, 48),
    ("DESCENT", 900, 0.30,  1200, 55),
    ("LANDING", 240, 0.35,   200, 30),
]


def mission_profile(t, scenario="standard"):
    """Return (phase, throttle, alt_m, isa_dev, tas) for mission-elapsed time t [s]."""
    acc = 0.0
    prev = MISSION_PHASES[0]
    for name, dur, thr, alt, tas in MISSION_PHASES:
        if t < acc + dur:
            f = (t - acc) / dur
            alt0 = prev[3]
            alt_i = alt0 + (alt - alt0) * min(1.0, f * 1.6)
            thr_i = thr
            if scenario == "hot_day":
                dev = 25.0
            elif scenario == "high_alt":
                alt_i *= 1.35
                dev = 5.0
            elif scenario == "transients":
                thr_i = np.clip(thr + 0.35 * math.sin(t / 9.0) + 0.2 * math.sin(t / 2.3), 0.12, 1.0)
                dev = 0.0
            else:
                dev = 0.0
            return name, float(thr_i), float(alt_i), dev, tas
        acc += dur
        prev = (name, dur, thr, alt, tas)
    return "LOITER", 0.48, 7000.0, 0.0, 48.0


TOTAL_MISSION_S = sum(p[1] for p in MISSION_PHASES)

In [ ]:
# Sanity check: fly one mission with a healthy engine and plot the profile.
eng = AeroPistonEngine(seed=0)
rows = []
t = 0.0
while t < TOTAL_MISSION_S:
    phase, thr, alt, dev, tas = mission_profile(t, 'standard')
    r = eng.step(60.0, thr, alt, dev, tas, deg_rate=1.0)
    r['phase'] = phase
    rows.append(r)
    t += 60.0
mis = pd.DataFrame(rows)
fig, ax = plt.subplots(2, 2, figsize=(13, 6))
ax[0, 0].plot(mis['alt_m']); ax[0, 0].set_title('altitude [m]')
ax[0, 1].plot(mis['rpm']); ax[0, 1].set_title('rpm')
ax[1, 0].plot(mis['egt'], label='EGT'); ax[1, 0].plot(mis['cht'], label='CHT')
ax[1, 0].legend(); ax[1, 0].set_title('temperatures [C]')
ax[1, 1].plot(mis['fuel_flow']); ax[1, 1].set_title('fuel flow [kg/h]')
for a in ax.ravel():
    a.set_xlabel('mission minute')
plt.tight_layout(); plt.show()
print('HEALTH_KEYS order (contract for module 4):', HEALTH_KEYS)

## 2. Run-to-failure corpus (30 units)

Identical logic to `ml/build_dataset.py` (kept in sync by hand — the feature
order `RAW + RESID + ['t']` is the deployment contract). Every unit is flown
mission after mission with a per-unit duty severity and a dominant degradation
mode until one health channel crosses the functional-failure threshold (0.80).

In [ ]:
RAW = ['rpm', 'map_kpa', 'fuel_flow', 'egt', 'cht', 'oil_p', 'oil_t',
       'coolant_t', 'vib', 'vib_1x', 'vib_half', 'bus_v', 'inj_act',
       'throttle', 'alt_m', 'oat_c']
RESID = ['r_rpm', 'r_fuel_flow', 'r_egt', 'r_cht', 'r_oil_p', 'r_oil_t',
         'r_vib', 'r_vib_half', 'r_bus_v', 'r_inj_act', 'r_bsfc']
FEATURES = RAW + RESID + ['t']          # 28 — ORDER IS THE CONTRACT
FAULT_CLASSES = ['NOMINAL', 'MISFIRE', 'INJECTOR_FOUL', 'COOLING_DEGRADED',
                 'LUBRICATION', 'BEARING_WEAR', 'SENSOR_DRIFT', 'COMBUSTION_INSTAB']
COMPONENTS = list(HEALTH_KEYS)
FAIL_THRESHOLD = 0.80
RUL_CAP = 600.0        # piecewise-linear RUL cap (C-MAPSS practice)
HORIZON_H = 100.0      # forecaster horizon [engine hours]
PRIORITIES = ['ROUTINE', 'SCHEDULED', 'URGENT', 'IMMEDIATE']


def label_fault(row):
    h = {k: row['h_' + k] for k in HEALTH_KEYS}
    if row['misfire'] > 0.18:
        return 'MISFIRE'
    dom, val = max(h.items(), key=lambda kv: kv[1])
    if val < 0.22:
        return 'NOMINAL'
    return {'injector_clog': 'INJECTOR_FOUL', 'cooling_fouling': 'COOLING_DEGRADED',
            'oil_degradation': 'LUBRICATION', 'bearing_wear': 'BEARING_WEAR',
            'sensor_drift': 'SENSOR_DRIFT', 'ignition_wear': 'COMBUSTION_INSTAB',
            'ring_wear': 'COMBUSTION_INSTAB'}[dom]


def simulate_unit(unit_id, rng, dt=60.0, sample_every=5, max_hours=2000.0):
    scenario = rng.choice(['standard', 'standard', 'hot_day', 'high_alt', 'transients'])
    seed = int(rng.integers(1e6))
    eng = AeroPistonEngine(seed=seed)
    ref = AeroPistonEngine(seed=seed + 1)          # healthy shadow twin
    for k in HEALTH_KEYS:
        eng.h[k] = float(abs(rng.normal(0, 0.015)))
    deg_rate = float(rng.uniform(0.7, 2.2))
    for k in HEALTH_KEYS:
        eng.rate_mult[k] = float(rng.uniform(0.35, 1.15))
    eng.rate_mult[str(rng.choice(HEALTH_KEYS))] = float(rng.uniform(1.8, 3.2))
    rows, t, i = [], 0.0, 0
    while eng.hours < max_hours:
        phase, thr, alt, dev, tas = mission_profile(t % TOTAL_MISSION_S, scenario)
        r = eng.step(dt, thr, alt, dev, tas, deg_rate=deg_rate)
        b = ref.step(dt, thr, alt, dev, tas, deg_rate=0.0)
        if i % sample_every == 0:
            for key in ['rpm', 'fuel_flow', 'egt', 'cht', 'oil_p', 'oil_t',
                        'vib', 'vib_half', 'bus_v', 'inj_act', 'bsfc']:
                r['r_' + key] = r[key] - b[key]
            r.pop('cylEGT', None); r.pop('cylCHT', None)
            r['unit'] = unit_id; r['phase'] = phase; r['scenario'] = scenario
            rows.append(r)
        t += dt; i += 1
        if max(eng.h.values()) >= FAIL_THRESHOLD:
            break
    df = pd.DataFrame(rows)
    df['RUL'] = df['t'].iloc[-1] - df['t']
    df['fault'] = df.apply(label_fault, axis=1)
    df['health_index'] = 100.0 * (1.0 - df[[f'h_{k}' for k in HEALTH_KEYS]].max(axis=1) / FAIL_THRESHOLD)
    return df

In [ ]:
N_UNITS = 30
rng = np.random.default_rng(7)
t0 = time.time()
frames = []
for u in range(N_UNITS):
    frames.append(simulate_unit(u, rng))
    print(f'unit {u:>2}  rows={len(frames[-1]):>6}  hours={frames[-1]["t"].iloc[-1]:7.1f}', end='\r')
df = pd.concat(frames, ignore_index=True)
print(f'\ncorpus: {len(df):,} rows / {N_UNITS} units in {time.time()-t0:.0f}s')
print(df.groupby('fault').size())
df.to_parquet('engine_runs.parquet')

### 2.1 EDA

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 7))
for u in df['unit'].unique()[:8]:
    d = df[df['unit'] == u]
    ax[0, 0].plot(d['t'], d['health_index'], lw=.8)
ax[0, 0].set_title('health index trajectories'); ax[0, 0].set_xlabel('engine hours')

for k in HEALTH_KEYS:
    d = df[df['unit'] == 0]
    ax[0, 1].plot(d['t'], d['h_' + k], lw=1, label=k)
ax[0, 1].legend(fontsize=6); ax[0, 1].set_title('unit 0 degradation modes')

df.groupby('fault').size().sort_values().plot.barh(ax=ax[0, 2])
ax[0, 2].set_title('fault label distribution')

ax[1, 0].hist(df['RUL'], bins=80); ax[1, 0].set_title('RUL [h] distribution')
sc = ax[1, 1].scatter(df['t'][::200], df['vib'][::200], c=df['h_bearing_wear'][::200], s=3, cmap='viridis')
ax[1, 1].set_title('vibration vs hours (colour = bearing wear)'); plt.colorbar(sc, ax=ax[1, 1])
ax[1, 2].scatter(df['r_cht'][::200], df['h_cooling_fouling'][::200], s=3, alpha=.3)
ax[1, 2].set_title('CHT residual vs cooling fouling')
plt.tight_layout(); plt.show()

corr = df[RESID + ['h_' + k for k in HEALTH_KEYS]].corr().loc[RESID, ['h_' + k for k in HEALTH_KEYS]]
plt.figure(figsize=(8, 5))
plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(HEALTH_KEYS)), HEALTH_KEYS, rotation=90)
plt.yticks(range(len(RESID)), RESID)
plt.colorbar(); plt.title('residual / degradation correlation'); plt.tight_layout(); plt.show()

## 3. Preprocessing, unit split and sequence windows

* residuals are EWMA-smoothed per unit (`alpha = 0.1`) exactly as every runtime does;
* normalisation is `(x - mean) / std` positionally over `FEATURES` — the same
  vectors are shipped in the bundle;
* the split is by **engine unit**, 20 % held out;
* sequence models consume windows of `SEQ_LEN = 60` samples (5 engine hours at
  5-minute sampling) with stride 10.

In [ ]:
ALPHA, SEQ_LEN, STRIDE = 0.1, 60, 10
df[RESID] = df.groupby('unit')[RESID].transform(lambda x: x.ewm(alpha=ALPHA).mean())
MEAN = df[FEATURES].mean().values
STD = (df[FEATURES].std() + 1e-3).values
X = np.clip((df[FEATURES].values - MEAN) / STD, -8, 8).astype('float32')

units = np.sort(df['unit'].unique())
n_test = max(1, int(round(0.20 * len(units))))
test_units = set(np.random.default_rng(2026).permutation(units)[:n_test].tolist())
is_test = df['unit'].isin(test_units).values
tr, te = ~is_test, is_test
unit_arr = df['unit'].values
t_arr = df['t'].values.astype('float64')
hi = df['health_index'].values
H = df[['h_' + k for k in HEALTH_KEYS]].values.astype('float32')
rul = np.minimum(df['RUL'].values, RUL_CAP).astype('float32')
y_cls = np.array([FAULT_CLASSES.index(c) for c in df['fault']])
print('held-out units:', sorted(test_units), '| rows train', tr.sum(), 'test', te.sum())


def windows(mask, seq_len=SEQ_LEN, stride=STRIDE):
    '''Window start indices that stay inside a single unit and inside `mask`.'''
    starts = []
    for u in np.unique(unit_arr):
        rows = np.flatnonzero(unit_arr == u)
        if not mask[rows[0]]:
            continue
        for s in range(0, len(rows) - seq_len, stride):
            starts.append(rows[s])
    return np.asarray(starts)


def seq_batch(starts, seq_len=SEQ_LEN):
    idx = starts[:, None] + np.arange(seq_len)[None, :]
    return X[idx]                                     # (n, seq_len, 28)


w_tr, w_te = windows(tr), windows(te)
print('windows train/test:', len(w_tr), len(w_te))

## Module 1 — sequence auto-encoder for anomaly detection

A **Conv1D encoder + LSTM decoder** auto-encoder is trained on *healthy windows
only* (health index > 92). Novelty is the per-window reconstruction MSE; the
alarm threshold is the **99.5th percentile of the healthy training error**, so
the design false-alarm rate is 0.5 % by construction. We then measure the
achieved detection rate on genuinely degraded held-out windows (health < 80)
and the achieved false-alarm rate on genuinely healthy held-out windows
(health > 96), plus the full ROC.

In [ ]:
hi_win = lambda starts: hi[starts + SEQ_LEN - 1]
healthy_tr = w_tr[hi_win(w_tr) > 92.0]
Xh = seq_batch(healthy_tr)
print('healthy training windows', Xh.shape)

inp = keras.Input((SEQ_LEN, len(FEATURES)))
z = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
z = layers.MaxPool1D(2)(z)
z = layers.Conv1D(32, 5, padding='same', activation='relu')(z)
z = layers.MaxPool1D(2)(z)
z = layers.LSTM(16, name='latent')(z)
d = layers.RepeatVector(SEQ_LEN)(z)
d = layers.LSTM(32, return_sequences=True)(d)
d = layers.Conv1D(64, 5, padding='same', activation='relu')(d)
out = layers.Conv1D(len(FEATURES), 1, activation='linear')(d)
ae = keras.Model(inp, out, name='seq_autoencoder')
ae.compile(keras.optimizers.Adam(1e-3), 'mse')
ae.summary()
ae.fit(Xh, Xh, epochs=25, batch_size=256, validation_split=0.1, verbose=2,
       callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

In [ ]:
def ae_err(starts, bs=2048):
    errs = []
    for i in range(0, len(starts), bs):
        b = seq_batch(starts[i:i + bs])
        errs.append(((ae.predict(b, verbose=0) - b) ** 2).mean(axis=(1, 2)))
    return np.concatenate(errs)


err_healthy = ae_err(healthy_tr)
THRESH = float(np.quantile(err_healthy, 0.995))
err_te = ae_err(w_te)
hi_te = hi_win(w_te)
ae_detection = float((err_te[hi_te < 80.0] > THRESH).mean())
ae_false_alarm = float((err_te[hi_te > 96.0] > THRESH).mean())
print(f'threshold={THRESH:.6f}  detection={ae_detection:.2%}  false-alarm={ae_false_alarm:.2%}')

# ROC over the threshold sweep (positives = degraded, negatives = healthy)
pos, neg = err_te[hi_te < 80.0], err_te[hi_te > 96.0]
grid = np.quantile(np.concatenate([pos, neg]), np.linspace(0, 1, 300))
tpr = [(pos > g).mean() for g in grid]
fpr = [(neg > g).mean() for g in grid]
auc = float(np.trapz(tpr[::-1], np.array(fpr[::-1])))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], 'k--', lw=.6)
ax[0].scatter([ae_false_alarm], [ae_detection], c='r', zorder=5, label='deployed threshold')
ax[0].set_xlabel('false alarm rate'); ax[0].set_ylabel('detection rate')
ax[0].set_title(f'ROC (AUC={auc:.4f})'); ax[0].legend()
ax[1].hist(neg, bins=60, alpha=.6, label='healthy', density=True)
ax[1].hist(pos, bins=60, alpha=.6, label='degraded', density=True)
ax[1].axvline(THRESH, c='r'); ax[1].set_xscale('log')
ax[1].legend(); ax[1].set_title('reconstruction error')
plt.tight_layout(); plt.show()

## Module 2 — CNN + GRU fault classifier

A dilated **Conv1D** stack extracts local signatures (misfire bursts in the 0.5x
vibration band, injector-driven fuel-flow residual steps) and a **GRU**
integrates them over the 5-hour window. Training uses class-balanced sample
weights because `NOMINAL` dominates a run-to-failure corpus. The reported
metric is **macro recall (balanced accuracy)** on held-out units.
Note `t` (hours since overhaul) is dropped: root cause must come from the
physics residuals, otherwise the model shortcuts diagnosis from engine age.

In [ ]:
CLF_IDX = [FEATURES.index(f) for f in RAW + RESID]          # excludes 't'
y_win_tr, y_win_te = y_cls[w_tr + SEQ_LEN - 1], y_cls[w_te + SEQ_LEN - 1]
Xw_tr, Xw_te = seq_batch(w_tr)[:, :, CLF_IDX], seq_batch(w_te)[:, :, CLF_IDX]
cnt = np.bincount(y_win_tr, minlength=len(FAULT_CLASSES)).astype('float32')
w_class = np.where(cnt > 0, cnt.sum() / np.maximum(cnt, 1) / (cnt > 0).sum(), 0.0)
sw = w_class[y_win_tr]

inp = keras.Input((SEQ_LEN, len(CLF_IDX)))
z = layers.Conv1D(64, 5, padding='causal', dilation_rate=1, activation='relu')(inp)
z = layers.BatchNormalization()(z)
z = layers.Conv1D(64, 5, padding='causal', dilation_rate=2, activation='relu')(z)
z = layers.Conv1D(64, 5, padding='causal', dilation_rate=4, activation='relu')(z)
z = layers.Bidirectional(layers.GRU(48))(z)
z = layers.Dropout(0.2)(z)
z = layers.Dense(64, activation='relu')(z)
out = layers.Dense(len(FAULT_CLASSES), activation='softmax')(z)
clf = keras.Model(inp, out, name='cnn_gru_classifier')
clf.compile(keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
clf.fit(Xw_tr, y_win_tr, sample_weight=sw, epochs=30, batch_size=256, verbose=2,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

In [ ]:
p_te = clf.predict(Xw_te, verbose=0)
pred = p_te.argmax(1)
clf_accuracy = float((pred == y_win_te).mean())
present = [c for c in range(len(FAULT_CLASSES)) if (y_win_te == c).any()]
recalls = {FAULT_CLASSES[c]: float((pred[y_win_te == c] == c).mean()) for c in present}
clf_balanced_acc = float(np.mean(list(recalls.values())))
print(f'held-out accuracy={clf_accuracy:.2%}  balanced (macro recall)={clf_balanced_acc:.2%}')
for k, v in recalls.items():
    print(f'  {k:<18} recall={v:.2%}')

cm = np.zeros((len(FAULT_CLASSES), len(FAULT_CLASSES)), int)
for a, b in zip(y_win_te, pred):
    cm[a, b] += 1
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(7, 6))
plt.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
for i in range(len(FAULT_CLASSES)):
    for j in range(len(FAULT_CLASSES)):
        if cm[i, j]:
            plt.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7)
plt.xticks(range(len(FAULT_CLASSES)), FAULT_CLASSES, rotation=90)
plt.yticks(range(len(FAULT_CLASSES)), FAULT_CLASSES)
plt.title('confusion matrix (row-normalised, held-out units)')
plt.colorbar(); plt.tight_layout(); plt.show()

## Module 3 — physics-informed RUL network

Plain regression on RUL ignores two facts we know from the plant model, so the
loss adds two physics terms:

\[
\mathcal{L} = \underbrace{\mathrm{MSE}(\hat{r}, r)}_{\text{supervised}}
 + \lambda_m \underbrace{\mathbb{E}\big[\max(0, \hat{r}_{t+\Delta} - \hat{r}_{t})^2\big]}_{\text{monotonicity penalty}}
 + \lambda_d \underbrace{\mathrm{MSE}\big(\hat{d}, \max_k h_k\big)}_{\text{degradation consistency}}
\]

* **Supervised term** — MSE on the piecewise-linear RUL target (capped at
  600 h, C-MAPSS practice) normalised to [0, 1].
* **Monotonicity penalty** — remaining useful life can only *decrease* along a
  unit's timeline (no self-healing). For each consecutive pair of windows from
  the same unit we penalise any predicted increase, squared and one-sided. This
  suppresses the saw-tooth predictions a plain MLP produces when the flight
  condition changes.
* **Degradation-consistency loss** — an auxiliary head predicts the observable
  damage state \(\max_k h_k\) (the same quantity the health index is built
  from). Sharing the trunk forces the latent representation to encode *damage*
  rather than *operating point*, which is what makes the RUL head extrapolate
  to unseen units instead of memorising duty cycles.

`lambda_m = 0.5`, `lambda_d = 0.3` (tuned on training units only).

In [ ]:
# Pairs of consecutive windows from the same unit -> monotonicity penalty.
PAIR_GAP = 1
w_tr_sorted = np.sort(w_tr)
pair_ok = np.isin(w_tr_sorted + STRIDE * PAIR_GAP, w_tr_sorted)
A = w_tr_sorted[pair_ok]
B = A + STRIDE * PAIR_GAP
XA, XB = seq_batch(A), seq_batch(B)
rA = (rul[A + SEQ_LEN - 1] / RUL_CAP).astype('float32')
dA = H[A + SEQ_LEN - 1].max(axis=1).astype('float32')
LAMBDA_M, LAMBDA_D = 0.5, 0.3


def trunk(seq_len=SEQ_LEN, n_feat=len(FEATURES)):
    inp = keras.Input((seq_len, n_feat))
    z = layers.Conv1D(48, 5, padding='same', activation='relu')(inp)
    z = layers.Conv1D(48, 5, padding='same', activation='relu')(z)
    z = layers.GRU(48)(z)
    z = layers.Dense(48, activation='relu')(z)
    rul_head = layers.Dense(1, activation='sigmoid', name='rul')(z)
    deg_head = layers.Dense(1, activation='sigmoid', name='deg')(z)
    return keras.Model(inp, [rul_head, deg_head], name='pinn_rul')


pinn = trunk()
opt = keras.optimizers.Adam(1e-3)


@tf.function
def pinn_step(xa, xb, ra, da):
    with tf.GradientTape() as tape:
        pa, dpa = pinn(xa, training=True)
        pb, _ = pinn(xb, training=True)
        mse = tf.reduce_mean(tf.square(tf.squeeze(pa, -1) - ra))
        mono = tf.reduce_mean(tf.square(tf.nn.relu(tf.squeeze(pb, -1) - tf.squeeze(pa, -1))))
        degc = tf.reduce_mean(tf.square(tf.squeeze(dpa, -1) - da))
        loss = mse + LAMBDA_M * mono + LAMBDA_D * degc
    opt.apply_gradients(zip(tape.gradient(loss, pinn.trainable_variables), pinn.trainable_variables))
    return loss, mse, mono, degc


ds = tf.data.Dataset.from_tensor_slices((XA, XB, rA, dA)).shuffle(8192).batch(256).prefetch(2)
for ep in range(20):
    agg = np.zeros(4)
    n = 0
    for xa, xb, ra, da in ds:
        vals = pinn_step(xa, xb, ra, da)
        agg += np.array([float(v) for v in vals]); n += 1
    print(f'epoch {ep+1:>2}  loss={agg[0]/n:.5f}  mse={agg[1]/n:.5f}  mono={agg[2]/n:.5f}  deg={agg[3]/n:.5f}')

In [ ]:
pred_rul = pinn.predict(seq_batch(w_te), verbose=0)[0].ravel() * RUL_CAP
true_rul = rul[w_te + SEQ_LEN - 1]
rul_mae_h = float(np.abs(pred_rul - true_rul).mean())
near = true_rul < 200
rul_mae_near_h = float(np.abs(pred_rul[near] - true_rul[near]).mean())
print(f'held-out RUL MAE={rul_mae_h:.1f} h   MAE within 200 h of failure={rul_mae_near_h:.1f} h')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(true_rul, pred_rul, s=3, alpha=.2)
ax[0].plot([0, RUL_CAP], [0, RUL_CAP], 'r--')
ax[0].set_xlabel('true RUL [h]'); ax[0].set_ylabel('predicted RUL [h]'); ax[0].set_title('parity')
u = sorted(test_units)[0]
m = (unit_arr[w_te + SEQ_LEN - 1] == u)
ax[1].plot(t_arr[w_te + SEQ_LEN - 1][m], true_rul[m], label='true')
ax[1].plot(t_arr[w_te + SEQ_LEN - 1][m], pred_rul[m], label='predicted')
ax[1].fill_between(t_arr[w_te + SEQ_LEN - 1][m], pred_rul[m] - rul_mae_h, pred_rul[m] + rul_mae_h, alpha=.2)
ax[1].set_title(f'held-out unit {u}: RUL track +/- sigma_h'); ax[1].legend()
ax[1].set_xlabel('engine hours'); plt.tight_layout(); plt.show()
sigma_h = rul_mae_h

## Module 4 — degradation trend forecaster (multi-head, 100 h horizon)

For every window we look **100 engine hours ahead inside the same unit** and
take the 7-component degradation vector at that point as the target; windows
whose unit history ends before the horizon are dropped. The model is a shared
Conv1D/GRU trunk with **7 sigmoid heads** (one per component, values in [0, 1]),
trained with per-head MSE. Reported metric: `fc_mae`, mean absolute error over
all components on held-out units.

In [ ]:
def lookahead_index(t_values, units, horizon_h):
    '''Index of the first row of the same unit with t >= t + horizon_h.'''
    n = len(t_values)
    idx = np.arange(n)
    valid = np.zeros(n, bool)
    for u in np.unique(units):
        rows = np.flatnonzero(units == u)
        ts = t_values[rows]
        j = np.searchsorted(ts, ts + horizon_h, side='left')
        ok = j < len(ts)
        idx[rows[ok]] = rows[j[ok]]
        valid[rows[ok]] = True
    return idx, valid


ahead, has_h = lookahead_index(t_arr, unit_arr, HORIZON_H)
Yf = H[ahead]                                    # 7 components, 100 h later
f_tr = w_tr[has_h[w_tr + SEQ_LEN - 1]]
f_te = w_te[has_h[w_te + SEQ_LEN - 1]]

inp = keras.Input((SEQ_LEN, len(FEATURES)))
z = layers.Conv1D(48, 5, padding='same', activation='relu')(inp)
z = layers.GRU(48)(z)
z = layers.Dense(64, activation='relu')(z)
heads = [layers.Dense(1, activation='sigmoid', name=f'h_{k}')(layers.Dense(16, activation='relu')(z))
         for k in COMPONENTS]
fcast = keras.Model(inp, layers.Concatenate()(heads), name='trend_forecaster')
fcast.compile(keras.optimizers.Adam(1e-3), 'mse', metrics=['mae'])
fcast.fit(seq_batch(f_tr), Yf[f_tr + SEQ_LEN - 1], epochs=25, batch_size=256,
          validation_split=0.1, verbose=2,
          callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

pf = fcast.predict(seq_batch(f_te), verbose=0)
yt = Yf[f_te + SEQ_LEN - 1]
fc_mae = float(np.abs(pf - yt).mean())
print(f'held-out forecaster MAE={fc_mae:.4f}')
for i, k in enumerate(COMPONENTS):
    print(f'  {k:<18} MAE={np.abs(pf[:, i] - yt[:, i]).mean():.4f}')

u = sorted(test_units)[0]
m = unit_arr[f_te + SEQ_LEN - 1] == u
plt.figure(figsize=(11, 4))
for i, k in enumerate(COMPONENTS[:3]):
    plt.plot(t_arr[f_te + SEQ_LEN - 1][m], yt[m, i], lw=1, label=f'{k} true(+100h)')
    plt.plot(t_arr[f_te + SEQ_LEN - 1][m], pf[m, i], lw=1, ls='--', label=f'{k} pred')
plt.legend(fontsize=7); plt.xlabel('engine hours'); plt.title(f'held-out unit {u}: 100 h-ahead forecast')
plt.tight_layout(); plt.show()

## Module 5 — maintenance advisory priority

The advisory head maps six *derived* scalars per component —
`value, rate_per_100h, forecast_100h, rul_norm, anomaly_norm, conf` — onto four
priority classes. Labels come from a documented deterministic doctrine
(`advisor_policy`), so module 5 is a **learned surrogate of that policy**: when
real maintenance outcomes (work orders, findings, removals) become available the
same head is retrained on those labels without changing the bundle contract or
the consuming UI.

In [ ]:
def advisor_policy(value, forecast, rul_h):
    '''IMMEDIATE if value>0.70 or rul<50h; URGENT if value>0.50 or forecast>0.80
    (failure crossed inside the horizon) or rul<150h; SCHEDULED if value>0.28 or
    rul<400h; else ROUTINE.'''
    p = np.zeros(len(value), 'int64')
    p[(value > 0.28) | (rul_h < 400.0)] = 1
    p[(value > 0.50) | (forecast > 0.80) | (rul_h < 150.0)] = 2
    p[(value > 0.70) | (rul_h < 50.0)] = 3
    return p


def lookback_index(t_values, units, horizon_h):
    idx = np.arange(len(t_values))
    for u in np.unique(units):
        rows = np.flatnonzero(units == u)
        ts = t_values[rows]
        j = np.clip(np.searchsorted(ts, ts - horizon_h, side='left'), 0, len(ts) - 1)
        idx[rows] = rows[j]
    return idx


back = lookback_index(t_arr, unit_arr, HORIZON_H)
# anomaly + confidence come from the *trained* modules 1 and 2 (deep versions)
row_of = lambda starts: starts + SEQ_LEN - 1
adv_rows = np.concatenate([w_tr, w_te])
anom = np.concatenate([ae_err(w_tr), ae_err(w_te)]) / max(THRESH, 1e-9)
conf = np.concatenate([clf.predict(seq_batch(w_tr)[:, :, CLF_IDX], verbose=0).max(1),
                       p_te.max(1)])
r = row_of(adv_rows)
keep = has_h[r]
feat, lab, uu = [], [], []
for ci in range(len(COMPONENTS)):
    value = H[r, ci]
    dt_h = np.maximum(t_arr[r] - t_arr[back[r]], 1e-6)
    rate = np.clip((value - H[back[r], ci]) / dt_h * HORIZON_H, 0, 1)
    fore = Yf[r, ci]
    feat.append(np.stack([value, rate, fore, np.minimum(rul[r], RUL_CAP) / RUL_CAP,
                          np.clip(anom, 0, 3) / 3.0, conf], 1)[keep])
    lab.append(advisor_policy(value, fore, rul[r])[keep])
    uu.append(unit_arr[r][keep])
Xa, ya, ua = np.concatenate(feat).astype('float32'), np.concatenate(lab), np.concatenate(uu)
a_te = np.isin(ua, list(test_units)); a_tr = ~a_te
cnt = np.bincount(ya[a_tr], minlength=4).astype('float32')
wc = np.where(cnt > 0, cnt.sum() / np.maximum(cnt, 1) / (cnt > 0).sum(), 0.0)

adv = keras.Sequential([keras.Input((6,)),
                        layers.Dense(48, activation='relu'),
                        layers.Dense(24, activation='relu'),
                        layers.Dense(4, activation='softmax')], name='advisor')
adv.compile(keras.optimizers.Adam(2e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
adv.fit(Xa[a_tr], ya[a_tr], sample_weight=wc[ya[a_tr]], epochs=40, batch_size=512, verbose=2)
ap = adv.predict(Xa[a_te], verbose=0).argmax(1)
advisor_acc = float((ap == ya[a_te]).mean())
print(f'held-out advisor accuracy={advisor_acc:.2%}')
for c, name in enumerate(PRIORITIES):
    m = ya[a_te] == c
    if m.any():
        print(f'  {name:<10} n={m.sum():>7}  recall={(ap[m] == c).mean():.2%}')

## Explainability — permutation importance and integrated gradients

*Permutation importance*: shuffle one input channel across the held-out set and
measure the drop in macro recall — model-agnostic and directly readable by
maintenance engineers. *Integrated gradients*: attribute a single diagnosis to
input channels by integrating the gradient along the straight path from a
healthy baseline window to the observed window, giving per-class evidence.

In [ ]:
# --- permutation importance on the fault classifier (macro recall drop) -----
def macro_recall(p, y):
    return float(np.mean([(p[y == c] == c).mean() for c in np.unique(y)]))


base = macro_recall(clf.predict(Xw_te, verbose=0).argmax(1), y_win_te)
sub = np.random.default_rng(0).choice(len(Xw_te), size=min(4000, len(Xw_te)), replace=False)
imp = []
for j, name in enumerate([FEATURES[i] for i in CLF_IDX]):
    Xp = Xw_te[sub].copy()
    Xp[:, :, j] = Xp[np.random.permutation(len(sub)), :, j]
    imp.append(base - macro_recall(clf.predict(Xp, verbose=0).argmax(1), y_win_te[sub]))
order = np.argsort(imp)
plt.figure(figsize=(7, 7))
plt.barh([[FEATURES[i] for i in CLF_IDX][k] for k in order], [imp[k] for k in order])
plt.xlabel('macro-recall drop when channel is permuted')
plt.title('permutation importance (fault classifier)'); plt.tight_layout(); plt.show()

# --- integrated gradients for one window, per predicted class ---------------
def integrated_gradients(model, x, baseline, steps=64, target=None):
    alphas = tf.linspace(0.0, 1.0, steps)
    path = baseline + alphas[:, None, None] * (x - baseline)
    with tf.GradientTape() as tape:
        tape.watch(path)
        out = model(path)
        out = out[:, target] if target is not None else tf.reduce_max(out, 1)
    grads = tape.gradient(out, path)
    return ((x - baseline) * tf.reduce_mean(grads, 0)).numpy()


healthy_baseline = Xw_te[y_win_te == FAULT_CLASSES.index('NOMINAL')][:64].mean(0, keepdims=True)
k = int(np.flatnonzero(y_win_te == np.bincount(y_win_te).argmax())[0])
x1 = tf.convert_to_tensor(Xw_te[k:k + 1])
cls = int(clf.predict(x1, verbose=0).argmax())
ig = integrated_gradients(clf, x1, tf.convert_to_tensor(healthy_baseline), target=cls)[0]
attr = np.abs(ig).sum(0)
o = np.argsort(attr)[::-1][:12]
plt.figure(figsize=(8, 4))
plt.bar(range(len(o)), attr[o])
plt.xticks(range(len(o)), [[FEATURES[i] for i in CLF_IDX][j] for j in o], rotation=60, ha='right')
plt.title(f'integrated gradients | predicted {FAULT_CLASSES[cls]}')
plt.tight_layout(); plt.show()

## Federated learning demo — FedAvg across 4 fleet clients

Operators rarely export raw engine telemetry. We therefore demonstrate
**FedAvg**: the 24 training units are partitioned into 4 simulated fleet
clients, each client trains the advisory/diagnosis head locally for E epochs,
and only the weights are averaged (weighted by client sample count) into the
global model. No telemetry leaves the client.

In [ ]:
N_CLIENTS, ROUNDS, LOCAL_EPOCHS = 4, 8, 2
train_units = np.array([u for u in units if u not in test_units])
shards = np.array_split(np.random.default_rng(3).permutation(train_units), N_CLIENTS)


def make_clf():
    m = keras.Sequential([keras.Input((SEQ_LEN, len(CLF_IDX))),
                          layers.Conv1D(32, 5, padding='causal', activation='relu'),
                          layers.GRU(32),
                          layers.Dense(32, activation='relu'),
                          layers.Dense(len(FAULT_CLASSES), activation='softmax')])
    m.compile(keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


global_model = make_clf()
client_sets = []
for sh in shards:
    st = w_tr[np.isin(unit_arr[w_tr + SEQ_LEN - 1], sh)]
    client_sets.append((seq_batch(st)[:, :, CLF_IDX], y_cls[st + SEQ_LEN - 1]))
    print('client units', sh, 'windows', len(st))

hist = []
for rnd in range(ROUNDS):
    weights, sizes = [], []
    gw = global_model.get_weights()
    for cx, cy in client_sets:
        local = make_clf()
        local.set_weights(gw)
        cnt = np.bincount(cy, minlength=len(FAULT_CLASSES)).astype('float32')
        w = np.where(cnt > 0, cnt.sum() / np.maximum(cnt, 1) / (cnt > 0).sum(), 0.0)
        local.fit(cx, cy, sample_weight=w[cy], epochs=LOCAL_EPOCHS, batch_size=256, verbose=0)
        weights.append(local.get_weights()); sizes.append(len(cx))
    tot = float(sum(sizes))
    global_model.set_weights([sum(w[i] * (s / tot) for w, s in zip(weights, sizes))
                              for i in range(len(gw))])
    acc = macro_recall(global_model.predict(Xw_te, verbose=0).argmax(1), y_win_te)
    hist.append(acc)
    print(f'FedAvg round {rnd+1}/{ROUNDS}  held-out macro recall={acc:.2%}')
plt.plot(range(1, ROUNDS + 1), hist, marker='o')
plt.xlabel('FedAvg round'); plt.ylabel('held-out macro recall')
plt.title('federated diagnosis head'); plt.grid(alpha=.3); plt.show()

## Knowledge distillation into the deployed students

The edge node, the NumPy backend runtime and the browser all execute the small
dense students defined in `ml/BUNDLE_SPEC.md`. Each deep teacher is distilled
into its student on the *last frame* of every window:

| module | student | teacher signal |
|--------|---------|----------------|
| autoencoder | 28-16-8-16-28, tanh | teacher reconstruction of the last frame |
| classifier | 27-64-32-8, relu | soft labels (temperature 2.0) |
| rul | 28-48-24-1, relu | teacher RUL (normalised) |
| forecaster | 28-32-7, relu | teacher 100 h-ahead vector |
| advisor | 6-24-4, relu | teacher priority soft labels |

Only `relu`, `tanh`, linear and softmax are allowed in the students — those are
the four kernels the NumPy/JS runtimes implement.

In [ ]:
T = 2.0
last_tr, last_te = row_of(w_tr), row_of(w_te)


def kd_loss(temp=T):
    def loss(y_soft, y_pred):
        return keras.losses.categorical_crossentropy(y_soft, y_pred)
    return loss


# --- student 1: dense auto-encoder 28-16-8-16-28 (tanh) ---------------------
ae_s = keras.Sequential([keras.Input((28,)),
                         layers.Dense(16, activation='tanh'),
                         layers.Dense(8, activation='tanh'),
                         layers.Dense(16, activation='tanh'),
                         layers.Dense(28, activation='linear')], name='ae_student')
ae_s.compile(keras.optimizers.Adam(2e-3), 'mse')
teacher_recon = ae.predict(Xh, verbose=0)[:, -1, :]
ae_s.fit(Xh[:, -1, :], teacher_recon, epochs=40, batch_size=512, verbose=0)
err_s = ((ae_s.predict(X[row_of(healthy_tr)], verbose=0) - X[row_of(healthy_tr)]) ** 2).mean(1)
THRESH_S = float(np.quantile(err_s, 0.995))
e_te = ((ae_s.predict(X[last_te], verbose=0) - X[last_te]) ** 2).mean(1)
s_det = float((e_te[hi[last_te] < 80] > THRESH_S).mean())
s_fa = float((e_te[hi[last_te] > 96] > THRESH_S).mean())
print(f'student AE: threshold={THRESH_S:.6f} detection={s_det:.2%} false-alarm={s_fa:.2%}')

# --- student 2: classifier 27-64-32-8 (relu) -------------------------------
soft = clf.predict(Xw_tr, verbose=0) ** (1 / T)
soft = soft / soft.sum(1, keepdims=True)
clf_s = keras.Sequential([keras.Input((len(CLF_IDX),)),
                          layers.Dense(64, activation='relu'),
                          layers.Dense(32, activation='relu'),
                          layers.Dense(8, activation='softmax')], name='clf_student')
clf_s.compile(keras.optimizers.Adam(2e-3), kd_loss())
cnt = np.bincount(y_win_tr, minlength=8).astype('float32')
wc2 = np.where(cnt > 0, cnt.sum() / np.maximum(cnt, 1) / (cnt > 0).sum(), 0.0)
clf_s.fit(X[last_tr][:, CLF_IDX], soft, sample_weight=wc2[y_win_tr],
          epochs=60, batch_size=512, verbose=0)
ps = clf_s.predict(X[last_te][:, CLF_IDX], verbose=0)
s_clf_acc = float((ps.argmax(1) == y_win_te).mean())
s_clf_bal = macro_recall(ps.argmax(1), y_win_te)
print(f'student CLF: accuracy={s_clf_acc:.2%} balanced={s_clf_bal:.2%}')

# --- student 3: RUL 28-48-24-1 (relu) --------------------------------------
rul_teacher = pinn.predict(seq_batch(w_tr), verbose=0)[0].ravel()
rul_s = keras.Sequential([keras.Input((28,)),
                          layers.Dense(48, activation='relu'),
                          layers.Dense(24, activation='relu'),
                          layers.Dense(1, activation='linear')], name='rul_student')
rul_s.compile(keras.optimizers.Adam(2e-3), 'mse')
rul_s.fit(X[last_tr], rul_teacher, epochs=60, batch_size=512, verbose=0)
hrs = np.clip(rul_s.predict(X[last_te], verbose=0).ravel(), 0, 1.1) * RUL_CAP
s_rul_mae = float(np.abs(hrs - rul[last_te]).mean())
nr = rul[last_te] < 200
s_rul_near = float(np.abs(hrs[nr] - rul[last_te][nr]).mean())
print(f'student RUL: MAE={s_rul_mae:.1f} h  near-failure MAE={s_rul_near:.1f} h')

# --- student 4: forecaster 28-32-7 (relu) ----------------------------------
fc_teacher = fcast.predict(seq_batch(f_tr), verbose=0)
fc_s = keras.Sequential([keras.Input((28,)),
                         layers.Dense(32, activation='relu'),
                         layers.Dense(7, activation='linear')], name='fc_student')
fc_s.compile(keras.optimizers.Adam(2e-3), 'mse')
fc_s.fit(X[row_of(f_tr)], fc_teacher, epochs=60, batch_size=512, verbose=0)
pfs = np.clip(fc_s.predict(X[row_of(f_te)], verbose=0), 0, 1)
s_fc_mae = float(np.abs(pfs - Yf[row_of(f_te)]).mean())
print(f'student forecaster: MAE={s_fc_mae:.4f}')

# --- student 5: advisor 6-24-4 (relu) --------------------------------------
soft_a = adv.predict(Xa[a_tr], verbose=0) ** (1 / T)
soft_a = soft_a / soft_a.sum(1, keepdims=True)
adv_s = keras.Sequential([keras.Input((6,)),
                          layers.Dense(24, activation='relu'),
                          layers.Dense(4, activation='softmax')], name='adv_student')
adv_s.compile(keras.optimizers.Adam(3e-3), kd_loss())
adv_s.fit(Xa[a_tr], soft_a, sample_weight=wc[ya[a_tr]], epochs=60, batch_size=512, verbose=0)
s_adv = float((adv_s.predict(Xa[a_te], verbose=0).argmax(1) == ya[a_te]).mean())
print(f'student advisor: accuracy={s_adv:.2%}')

### Export `dt_models.json` (schema 2)

In [ ]:
def dense_params(model):
    '''Flat [W, b, W, b, ...] row-major nested lists, fan_in x fan_out.'''
    out = []
    for l in model.layers:
        w = l.get_weights()
        if len(w) == 2:
            out += [np.round(w[0], 6).tolist(), np.round(w[1], 6).tolist()]
    return out


bundle = {
    'schema': 2,
    'created_at': datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'trained_rows': int(tr.sum()),
    'trained_units': int(len(units) - len(test_units)),
    'features': FEATURES,
    'classes': FAULT_CLASSES,
    'components': COMPONENTS,
    'norm': {'mean': [round(float(v), 6) for v in MEAN],
             'std': [round(float(v), 6) for v in STD]},
    'resid_alpha': ALPHA,
    'autoencoder': {'params': dense_params(ae_s), 'act': 'tanh',
                    'threshold': round(THRESH_S, 6), 'score_scale': 1.0},
    'classifier': {'params': dense_params(clf_s), 'act': 'relu',
                   'feature_idx': CLF_IDX, 'features': [FEATURES[i] for i in CLF_IDX]},
    'rul': {'params': dense_params(rul_s), 'act': 'relu', 'scale': RUL_CAP,
            'sigma_h': round(s_rul_mae, 3)},
    'forecaster': {'params': dense_params(fc_s), 'act': 'relu',
                   'horizon_h': HORIZON_H, 'components': COMPONENTS},
    'advisor': {'params': dense_params(adv_s), 'act': 'relu',
                'inputs': ['value', 'rate_per_100h', 'forecast_100h', 'rul_norm',
                           'anomaly_norm', 'conf'],
                'priorities': PRIORITIES},
    'metrics': {'ae_detection': round(s_det, 4), 'ae_false_alarm': round(s_fa, 4),
                'clf_accuracy': round(s_clf_acc, 4), 'clf_balanced_acc': round(s_clf_bal, 4),
                'rul_mae_h': round(s_rul_mae, 2), 'rul_mae_near_h': round(s_rul_near, 2),
                'fc_mae': round(s_fc_mae, 4), 'advisor_acc': round(s_adv, 4)},
    'held_out_units': int(len(test_units)),
}
with open('dt_models.json', 'w') as f:
    json.dump(bundle, f)
print('bundle', os.path.getsize('dt_models.json') / 1024, 'kB')
print(json.dumps(bundle['metrics'], indent=2))

try:
    from google.colab import files
    files.download('dt_models.json')
except Exception as e:
    print('not running in Colab, skipping download:', e)

## ONNX export + INT8 quantisation for the edge node

The edge node (companion computer) runs onnxruntime. We export the *deep*
teachers to ONNX and dynamically quantise them to INT8 — roughly 4x smaller and
2-3x faster on ARM CPUs — then verify the quantised outputs against the float
model on held-out windows before shipping.

In [ ]:
import tf2onnx, onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

os.makedirs('onnx', exist_ok=True)
exports = {'autoencoder': (ae, (None, SEQ_LEN, len(FEATURES))),
           'classifier': (clf, (None, SEQ_LEN, len(CLF_IDX))),
           'forecaster': (fcast, (None, SEQ_LEN, len(FEATURES))),
           'advisor': (adv, (None, 6))}
for name, (model, shape) in exports.items():
    spec = (tf.TensorSpec(shape, tf.float32, name='input'),)
    onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=17)
    fp32 = f'onnx/{name}.onnx'
    int8 = f'onnx/{name}.int8.onnx'
    onnx.save(onnx_model, fp32)
    quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)
    print(f'{name:<12} fp32={os.path.getsize(fp32)/1024:7.0f} kB  int8={os.path.getsize(int8)/1024:7.0f} kB')

# --- parity check: float Keras vs INT8 onnxruntime on held-out windows -----
xb = Xw_te[:256].astype('float32')
sess = ort.InferenceSession('onnx/classifier.int8.onnx', providers=['CPUExecutionProvider'])
q = sess.run(None, {sess.get_inputs()[0].name: xb})[0]
f = clf.predict(xb, verbose=0)
print('INT8 vs FP32 top-1 agreement:', float((q.argmax(1) == f.argmax(1)).mean()))
print('mean abs prob delta:', float(np.abs(q - f).mean()))

## Dropping the bundle into the running system

This notebook was generated against the AeroTwin-DT repo layout. In **this**
delivered build (FastAPI + PostgreSQL backend, React + Tailwind frontend) the
trained bundle is consumed from exactly two places:

1. Download `dt_models.json` from this notebook (the `files.download` cell above).
2. Copy it into the repository in **both** places:

```bash
cp ~/Downloads/dt_models.json  AeroTwin-DT/backend/app/assets/dt_models.json
cp ~/Downloads/dt_models.json  AeroTwin-DT/frontend/public/assets/dt_models.json
```

3. Restart the backend (`uvicorn app.main:app --reload`) — it loads the bundle
   once at process start (`app/ml_bundle.py`) and logs the feature/class count
   and autoencoder threshold on success. `GET /api/model-meta` will then report
   the real `metrics` block from this notebook (AE detection rate, classifier
   accuracy, RUL MAE, etc.), which is what the **Architecture** view in the
   dashboard displays.
4. Validate the contract before deploying — a mismatched `features` list
   between this notebook and `backend/app/ml_bundle.py`'s `RAW_KEYS` /
   `RESID_KEYS` will silently misalign the feature vector. Re-run the
   `assert` cell above if you changed the feature engineering here.
